# Binary Trees Fundamentals

---

## The Big WHY

Trees introduce *hierarchy* — a shape that mirrors how a huge amount of real data is actually organized:

- File systems (folders inside folders)
- JSON/XML documents (nested keys)
- Decision Trees and Random Forests in ML
- HTML DOM
- Org charts

More importantly: trees are the backbone of **efficient search**. A sorted array gives you O(n) insert. A hash map gives you O(1) lookup but no ordering. A Binary Search Tree gives you O(log n) search *and* keeps things ordered. That tradeoff will matters a lot in many areas and to build vector retrieval systems.

---

## Vocabulary

```
        1          ← root (no parent)
       / \
      2   3        ← internal nodes
     / \
    4   5          ← leaves (no children)
```

| Term | Meaning |
|------|--------|
| **Root** | Top node, no parent |
| **Leaf** | Node with no children |
| **Height** | Longest path from root to a leaf |
| **Depth** | Distance from root to a specific node |
| **Subtree** | Any node + everything below it |
| **Binary tree** | Each node has at most 2 children (left, right) |

The tree above has **height = 2** (root → 2 → 4, two edges).

---

## The Three Traversals — and Why Each Exists

Traversal = visiting every node in some order. There are three classic orders, each useful for different things:

| Traversal | Order | Trick to remember | Real use |
|-----------|-------|-------------------|----------|
| **Inorder** | Left → Node → Right | "In" = in sequence | BST gives sorted output |
| **Preorder** | Node → Left → Right | "Pre" = process me first | Copying/serializing a tree |
| **Postorder** | Left → Right → Node | "Post" = process me last | Deleting a tree, calculating sizes |

On the tree above:
- Inorder: 4, 2, 5, 1, 3
- Preorder: 1, 2, 4, 5, 3
- Postorder: 4, 5, 2, 3, 1

---

## Part 1 — Build the TreeNode

A node needs three things: its value, a pointer to its left child, and a pointer to its right child. Children default to `None`.

In [2]:
# Define TreeNode with val, left, right
# Add a __repr__ so printing a node shows something useful


# Build this tree manually by wiring up nodes:
#        1
#       / \
#      2   3
#     / \
#    4   5

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


# building the tree manually
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.left.left = TreeNode(4)
root.left.right = TreeNode(5)



## Part 2 — The Three Recursive Traversals

Every recursive tree function has the same skeleton:

```
def traverse(node):
    if node is None:       ← base case (always first)
        return
    # do something with node.val
    traverse(node.left)
    traverse(node.right)
```

The *only* thing that changes between inorder/preorder/postorder is **where you put the 'do something' line**.

In [3]:
# inorder(node, result=None) → Left, Node, Right
def inorder(node, result=None):

    if result is None:
        result = []

    # base case
    if node is None:
        return result
    
    inorder(node.left, result)
    result.append(node.val)
    inorder(node.right, result)
    return result


# preorder(node, result=None) → Node, Left, Right
def preorder(node, result=None):
    if result is None:
        result = []

    if node is None:
        return result
    
    result.append(node.val)
    preorder(node.left, result)
    preorder(node.right, result)
    return result

# postorder(node, result=None) → Left, Right, Node
def postorder(node, result=None):
    if result is None:
        result = []

    if node is None:
        return result
    
    postorder(node.left, result)
    postorder(node.right, result)
    result.append(node.val)
    return result

# test
print("Inorder:  ", inorder(root))    # [4, 2, 5, 1, 3]
print("Preorder: ", preorder(root))   # [1, 2, 4, 5, 3]
print("Postorder:", postorder(root))  # [4, 5, 2, 3, 1]


Inorder:   [4, 2, 5, 1, 3]
Preorder:  [1, 2, 4, 5, 3]
Postorder: [4, 5, 2, 3, 1]


## Part 3 — Height and Count

The recursive insight for height:
> *height = 1 + max(height of left subtree, height of right subtree)*

Decide: what should `height(None)` return so that a single leaf node gives height 0?

In [ ]:
# height(node) → longest edge path from node to a leaf
# height calculates the edges so for a leaf node it should be 0, so we do return -1 in base case so 1 + max(-1, -1) becomes 0
def height(node):
    if node is None:   # base case, height(None) should return -1 so a single leaf comes out as 0
        return -1
    return 1 + max(height(node.left), height(node.right))

# count_nodes(node) → total number of nodes in the tree
# we are counting nodes here like in depth, so we return 0 in base case
def count_nodes(node):   
    if node is None:    # base case
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

# test
print("Height:", height(root))         # 2
print("Node count:", count_nodes(root)) # 5

Height: 2
Node count: 5


## Part 4 — Build Tree from List (LeetCode helper)

LeetCode gives trees as level-order lists like `[1, 2, 3, None, None, 4, 5]`. Build these two helpers now to paste them into every tree problem.

**Hint:** `build_tree` uses a queue. Process nodes level by level, assigning left and right children from the list.

In [5]:
from collections import deque

# build_tree(values) → root TreeNode
# values is a level-order list, None means no child
def build_tree(values):
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft() 

        # left child
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1    # always increment i, even if we we skip to match the index

        # right child
        if values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

    

# tree_to_list(root) → level-order list (for verifying your answers)
# strip trailing Nones at the end
def tree_to_list(root):
    queue = deque([root])
    result = []
    while queue:
        node = queue.popleft()
        if node is not None:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)  # represent the missing nodes, part of the structure
    
    # remove the trailing nones because they are just empty padding
    while result and result[-1] is None:
        result.pop()
    return result


# test
t = build_tree([1, 2, 3, 4, 5])
print(tree_to_list(t))   # [1, 2, 3, 4, 5]
print(inorder(t))        # [4, 2, 5, 1, 3]

# with a None in the middle
t2 = build_tree([1, 2, 3, None, None, 4, 5])
print(tree_to_list(t2))  # [1, 2, 3, None, None, 4, 5]


[1, 2, 3, 4, 5]
[4, 2, 5, 1, 3]
[1, 2, 3, None, None, 4, 5]


---

## LeetCode

### Problem 1: Invert Binary Tree (LC #226)

**What it asks:** Flip the tree left-right (mirror image).

**Think first:** At every node, swap left and right children, then do the same below. Which traversal order does that suggest — do you swap *before* or *after* recursing into children?

```
Input:  [4, 2, 7, 1, 3, 6, 9]
Output: [4, 7, 2, 9, 6, 3, 1]
```

In [8]:
# invertTree(root) → root
# Base case: None → return None
# Recurse into both children, then swap left/right (postorder traverse)
def invertTree(root):
    if root is None:
        return None
    
    invertTree(root.left)
    invertTree(root.right)
    root.left, root.right = root.right, root.left
    return root

## test cases
t = build_tree([4, 2, 7, 1, 3, 6, 9])
print(tree_to_list(t))          # [4, 2, 7, 1, 3, 6, 9]
invertTree(t)
print(tree_to_list(t))          # [4, 7, 2, 9, 6, 3, 1]

# edge cases
print(invertTree(None))         # None
print(tree_to_list(invertTree(build_tree([1]))))  # [1]


[4, 2, 7, 1, 3, 6, 9]
[4, 7, 2, 9, 6, 3, 1]
None
[1]


### time complexity
- we visit every node once, so it is O(n)

### Space complexity
- O(h) where h is the height.  
for best case when the tree is balanced, height is log n, so it is O(log n),  
for the worst case i.e skewed tree(every node only has one child, like a linked list) , height is n, so it is O(n)


---

### Problem 2: Maximum Depth of Binary Tree (LC #104)

**What it asks:** Number of nodes along the longest root-to-leaf path.

**This is almost identical to `height()` from Part 3** — the only difference is the convention: depth counts *nodes*, height counts *edges*. So what should the base case return here?

```
Input:  [3, 9, 20, None, None, 15, 7]
Output: 3
```

In [9]:
# maxDepth(root) → int
# One-liner body after the base case
def maxDepth(root):
    # we are counting the max number of nodes from root to leaf here
    # if it was height we would be counting max number of edges with base case -1
    if root is None:
        return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))

# test cases
print(maxDepth(build_tree([3, 9, 20, None, None, 15, 7])))  # 3
print(maxDepth(build_tree([1, None, 2])))                    # 2
print(maxDepth(None))                                        # 0
print(maxDepth(build_tree([1])))                             # 1


3
2
0
1


---

## ML Connection — Where Trees Show Up

**Decision Tree (sklearn):** Each internal node is a feature split (`feature_value < threshold?`). `predict()` traverses root → leaf. Understanding traversal = understanding what your model does under the hood.

**Random Forest:** N decision trees, each trained on a bootstrap sample. `predict()` averages the leaf values. Tree *height* directly controls overfitting — deeper trees overfit more.

**XGBoost/LightGBM:** Gradient-boosted trees — each new tree corrects the residuals of all previous ones. Same traversal logic, different training objective.

---

## Summary

| Concept | Key idea |
|---------|----------|
| TreeNode | val + left + right |
| Inorder | L → Node → R (BST gives sorted order) |
| Preorder | Node → L → R (serialize/copy) |
| Postorder | L → R → Node (calculate from leaves up) |
| Height | 1 + max(height(left), height(right)) |
| Base case | Always `if node is None: return <something>` , for height it is -1 and for depth it is 0|
| Both LC problems | Same recursive skeleton — only the return value differs |
